In [1]:
import requests
import psycopg2
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from tqdm import tqdm
import subprocess
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv()

# Mantener la Mac despierta mientras corre el script (macOS)
caffeinate_process = subprocess.Popen(['caffeinate'])

# Configuración conexión PostgreSQL desde .env
dbname = os.getenv("POSTGRES_DB")
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")
port = os.getenv("POSTGRES_PORT")

API_KEY = os.getenv("FMP_API_KEY")

# Conexión a la base de datos
conn = psycopg2.connect(
    dbname=dbname,
    user=user,
    password=password,
    host=host,
    port=port
)
cursor = conn.cursor()


def obtener_tickers():
    try:
        cursor.execute("SELECT ticker FROM infraestructura.tickers_test_api_raw")
        tickers = cursor.fetchall()
        return [t[0] for t in tickers]
    except Exception as e:
        print(f"Error al obtener tickers: {e}")
        return []

def obtener_ultimo_ano(ticker):
    query = "SELECT MAX(calendarYear) FROM api_raw.ratios_historicos_anual WHERE ticker = %s;"
    cursor.execute(query, (ticker,))
    result = cursor.fetchone()
    return result[0] if result and result[0] else 0

def redondear(valor, decimales=6):
    return round(valor, decimales) if isinstance(valor, (int, float)) else None

def log_update(schema_name, table_name, ticker, status, message):
    try:
        cursor.execute("""
            INSERT INTO infraestructura.update_logs (schema_name, table_name, ticker, status, message)
            VALUES (%s, %s, %s, %s, %s);
        """, (schema_name, table_name, ticker, status, message))
        conn.commit()
    except Exception as e:
        print(f"Error al registrar log para {ticker}: {e}")
        conn.rollback()

def obtener_ratios_financieros_filtrados(ticker, ultimo_ano):
    try:
        url = f"https://financialmodelingprep.com/api/v3/ratios/{ticker}?period=annual&apikey={API_KEY}"
        response = requests.get(url)
        if response.status_code != 200:
            print(f"Error al obtener ratios para {ticker}: {response.status_code}")
            return []
        data = response.json()
        if not data:
            print(f"No hay datos de ratios para {ticker}")
            return []

        ratios = []
        for reporte in data:
            year = int(reporte.get('calendarYear', 0))
            if year > ultimo_ano and year >= 2010:
                ratios.append({
                    'ticker': ticker,
                    'date': reporte.get('date'),  # YYYY-MM-DD
                    'calendarYear': year,
                    'period': reporte.get('period'),
                    'currentRatio': redondear(reporte.get('currentRatio')),
                    'quickRatio': redondear(reporte.get('quickRatio')),
                    'cashRatio': redondear(reporte.get('cashRatio')),
                    'grossProfitMargin': redondear(reporte.get('grossProfitMargin')),
                    'operatingProfitMargin': redondear(reporte.get('operatingProfitMargin')),
                    'netProfitMargin': redondear(reporte.get('netProfitMargin')),
                    'returnOnEquity': redondear(reporte.get('returnOnEquity')),
                    'returnOnCapitalEmployed': redondear(reporte.get('returnOnCapitalEmployed')),
                    'debtEquityRatio': redondear(reporte.get('debtEquityRatio')),
                    'interestCoverage': redondear(reporte.get('interestCoverage')),
                    'cashFlowToDebtRatio': redondear(reporte.get('cashFlowToDebtRatio')),
                    'freeCashFlowPerShare': redondear(reporte.get('freeCashFlowPerShare')),
                    'cashPerShare': redondear(reporte.get('cashPerShare')),
                    'payoutRatio': redondear(reporte.get('payoutRatio')),
                    'priceToBookRatio': redondear(reporte.get('priceToBookRatio')),
                    'priceToSalesRatio': redondear(reporte.get('priceToSalesRatio')),
                    'priceEarningsRatio': redondear(reporte.get('priceEarningsRatio')),
                    'priceFairValue': redondear(reporte.get('priceFairValue'))
                })
        return ratios

    except Exception as e:
        print(f"Error al obtener ratios para {ticker}: {e}")
        return []

def insertar_ratios_en_base_de_datos(datos_lista):
    if not datos_lista:
        return
    query = """
    INSERT INTO api_raw.ratios_historicos_anual (
        ticker, date, calendarYear, period, currentRatio, quickRatio, cashRatio, grossProfitMargin, 
        operatingProfitMargin, netProfitMargin, returnOnEquity, returnOnCapitalEmployed, debtEquityRatio, 
        interestCoverage, cashFlowToDebtRatio, freeCashFlowPerShare, cashPerShare, payoutRatio, 
        priceToBookRatio, priceToSalesRatio, priceEarningsRatio, priceFairValue
    )
    VALUES (
        %(ticker)s, %(date)s, %(calendarYear)s, %(period)s, %(currentRatio)s, %(quickRatio)s, %(cashRatio)s, %(grossProfitMargin)s, 
        %(operatingProfitMargin)s, %(netProfitMargin)s, %(returnOnEquity)s, %(returnOnCapitalEmployed)s, %(debtEquityRatio)s, 
        %(interestCoverage)s, %(cashFlowToDebtRatio)s, %(freeCashFlowPerShare)s, %(cashPerShare)s, %(payoutRatio)s, 
        %(priceToBookRatio)s, %(priceToSalesRatio)s, %(priceEarningsRatio)s, %(priceFairValue)s
    )
    ON CONFLICT (ticker, date) DO UPDATE SET updated_at = CURRENT_TIMESTAMP;
    """
    try:
        cursor.executemany(query, datos_lista)
        conn.commit()
        print(f"{len(datos_lista)} filas insertadas o actualizadas.")
    except Exception as e:
        print(f"Error al insertar datos: {e}")
        conn.rollback()

def ejecutar_actualizacion_incremental():
    start_time = time.time()
    tickers = obtener_tickers()
    if not tickers:
        print("No se encontraron tickers.")
        return

    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {}
        for ticker in tickers:
            ultimo_ano = obtener_ultimo_ano(ticker)
            futures[executor.submit(obtener_ratios_financieros_filtrados, ticker, ultimo_ano)] = ticker

        for future in tqdm(as_completed(futures), total=len(futures), desc="Actualizando ratios anuales"):
            ticker = futures[future]
            try:
                datos = future.result()
                if datos:
                    insertar_ratios_en_base_de_datos(datos)
                    log_update('api_raw', 'ratios_historicos_anual', ticker, 'success', 'Carga completa')
                else:
                    log_update('api_raw', 'ratios_historicos_anual', ticker, 'success', 'Sin datos nuevos')
            except Exception as e:
                print(f"Error en ticker {ticker}: {e}")
                log_update('api_raw', 'ratios_historicos_anual', ticker, 'fail', str(e))

    end_time = time.time()
    caffeinate_process.terminate()
    print(f"Tiempo total: {end_time - start_time:.2f} segundos.")
    cursor.close()
    conn.close()

if __name__ == "__main__":
    ejecutar_actualizacion_incremental()


Actualizando ratios anuales: 100%|████████████████| 5/5 [00:01<00:00,  4.85it/s]

Tiempo total: 1.05 segundos.
